<a href="https://colab.research.google.com/github/Harshita-code31/24cd3016_cnn_lab/blob/main/cifar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

:

In [1]:
!pip install timm -q

In [2]:
%%writefile dataset.py

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np

DATA_DIR   = "./data"
BATCH_SIZE = 64
IMG_SIZE   = 224
SEED       = 42

CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

torch.manual_seed(SEED)
np.random.seed(SEED)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        if self.transform:
            img = self.transform(img)
        return img, label

def get_loaders():
    full_train = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=None)
    test_ds    = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_transform)

    n_total = len(full_train)
    n_val   = int(n_total * 0.2)
    n_train = n_total - n_val

    indices = list(range(n_total))
    np.random.shuffle(indices)
    train_idx = indices[:n_train]
    val_idx   = indices[n_train:]

    train_ds = TransformSubset(full_train, train_idx, train_transform)
    val_ds   = TransformSubset(full_train, val_idx,   eval_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
    return train_loader, val_loader, test_loader

Writing dataset.py


In [3]:
%%writefile models.py

import timm

NUM_CLASSES = 10

def get_model(name):
    if name == "resnet":
        return timm.create_model("resnet50", pretrained=True, num_classes=NUM_CLASSES)
    elif name == "efficientnet":
        return timm.create_model("efficientnet_b0", pretrained=True, num_classes=NUM_CLASSES)
    elif name == "convnext":
        return timm.create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES)
    elif name == "vit":
        return timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
    else:
        raise ValueError(f"Unknown model: {name}")

Writing models.py


In [4]:
%%writefile train.py

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import os, json
from dataset import get_loaders
from models import get_model

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS      = 5
LR          = 1e-4
SAVE_DIR    = "./checkpoints"
MODEL_NAMES = ["resnet", "efficientnet", "convnext", "vit"]

os.makedirs(SAVE_DIR, exist_ok=True)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

def train_model(name):
    print(f"\n{'='*50}\n  Training: {name.upper()}\n{'='*50}")
    train_loader, val_loader, _ = get_loaders()
    model     = get_model(name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=LR)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = validate(model, val_loader, criterion)
        scheduler.step()
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)
        print(f"  Epoch {epoch}/{EPOCHS} | Train Acc: {tr_acc:.4f} | Val Acc: {vl_acc:.4f}")
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), f"{SAVE_DIR}/{name}_best.pth")
            print(f"  ✓ Best model saved (val_acc={best_val_acc:.4f})")

    with open(f"{SAVE_DIR}/{name}_history.json", "w") as f:
        json.dump(history, f)
    return history

if __name__ == "__main__":
    print(f"Device: {DEVICE}")
    for name in MODEL_NAMES:
        train_model(name)
    print("\n All models trained!")

Writing train.py


In [5]:
!python train.py

Device: cuda

  Training: RESNET
100% 170M/170M [00:23<00:00, 7.14MB/s]
Train: 40,000 | Val: 10,000 | Test: 10,000
model.safetensors: 100% 102M/102M [00:01<00:00, 69.2MB/s]
  Epoch 1/5 | Train Acc: 0.7914 | Val Acc: 0.9431
  ✓ Best model saved (val_acc=0.9431)
  Epoch 2/5 | Train Acc: 0.9357 | Val Acc: 0.9559
  ✓ Best model saved (val_acc=0.9559)
  Epoch 3/5 | Train Acc: 0.9548 | Val Acc: 0.9600
  ✓ Best model saved (val_acc=0.9600)
  Epoch 4/5 | Train Acc: 0.9640 | Val Acc: 0.9622
  ✓ Best model saved (val_acc=0.9622)
  Epoch 5/5 | Train Acc: 0.9671 | Val Acc: 0.9625
  ✓ Best model saved (val_acc=0.9625)

  Training: EFFICIENTNET
Train: 40,000 | Val: 10,000 | Test: 10,000
model.safetensors: 100% 21.4M/21.4M [00:00<00:00, 21.4MB/s]
  Epoch 1/5 | Train Acc: 0.8065 | Val Acc: 0.9168
  ✓ Best model saved (val_acc=0.9168)
  Epoch 2/5 | Train Acc: 0.9352 | Val Acc: 0.9438
  ✓ Best model saved (val_acc=0.9438)
  Epoch 3/5 | Train Acc: 0.9619 | Val Acc: 0.9536
  ✓ Best model saved (val_acc=0.

In [6]:
import os
files = os.listdir("./checkpoints")
print(files)

['resnet_best.pth', 'efficientnet_best.pth', 'efficientnet_history.json', 'resnet_history.json', 'convnext_best.pth']


In [9]:
%%writefile evaluate.py

import torch, json, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from dataset import get_loaders, CLASSES
from models import get_model

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR    = "./checkpoints"
MODEL_NAMES = ["resnet", "efficientnet", "convnext"]  # skipping vit
os.makedirs("./outputs", exist_ok=True)

def evaluate_model(name):
    _, _, test_loader = get_loaders()
    model = get_model(name).to(DEVICE)
    model.load_state_dict(torch.load(f"{SAVE_DIR}/{name}_best.pth", map_location=DEVICE))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            outputs = model(imgs.to(DEVICE))
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()

    print(f"\n{'='*50}")
    print(f"  {name.upper()}  —  Test Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(classification_report(all_labels, all_preds, target_names=CLASSES))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(f"Confusion Matrix — {name.upper()}")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(f"./outputs/{name}_confusion_matrix.png", dpi=120)
    plt.show()
    return acc

def plot_comparison(results):
    names = list(results.keys())
    accs  = [results[n] * 100 for n in names]
    plt.figure(figsize=(8, 5))
    bars = plt.bar(names, accs, color=["#4C72B0", "#DD8452", "#55A868"])
    plt.ylim(0, 100)
    plt.ylabel("Test Accuracy (%)")
    plt.title("Model Comparison — CIFAR-10 Test Accuracy")
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{acc:.2f}%", ha="center", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig("./outputs/model_comparison.png", dpi=120)
    plt.show()

def plot_training_curves():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, name in enumerate(MODEL_NAMES):
        with open(f"{SAVE_DIR}/{name}_history.json") as f:
            h = json.load(f)
        epochs = range(1, len(h["train_acc"]) + 1)
        axes[i].plot(epochs, h["train_acc"], label="Train", marker="o")
        axes[i].plot(epochs, h["val_acc"],   label="Val",   marker="s")
        axes[i].set_title(name.upper())
        axes[i].set_xlabel("Epoch")
        axes[i].set_ylabel("Accuracy")
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    plt.suptitle("Training vs Validation Accuracy", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("./outputs/training_curves.png", dpi=120)
    plt.show()

results = {}
for name in MODEL_NAMES:
    results[name] = evaluate_model(name)

plot_comparison(results)
plot_training_curves()

Writing evaluate.py


In [10]:
!python evaluate.py

Train: 40,000 | Val: 10,000 | Test: 10,000

  RESNET  —  Test Accuracy: 0.9661
              precision    recall  f1-score   support

    airplane       0.97      0.98      0.97      1000
  automobile       0.98      0.98      0.98      1000
        bird       0.97      0.97      0.97      1000
         cat       0.94      0.92      0.93      1000
        deer       0.96      0.97      0.97      1000
         dog       0.94      0.94      0.94      1000
        frog       0.98      0.99      0.98      1000
       horse       0.98      0.96      0.97      1000
        ship       0.98      0.98      0.98      1000
       truck       0.98      0.97      0.98      1000

    accuracy                           0.97     10000
   macro avg       0.97      0.97      0.97     10000
weighted avg       0.97      0.97      0.97     10000

Figure(1000x800)
Train: 40,000 | Val: 10,000 | Test: 10,000

  EFFICIENTNET  —  Test Accuracy: 0.9527
              precision    recall  f1-score   support

    a

In [11]:
%%writefile predict.py

import torch, sys
from torchvision import transforms
from PIL import Image
from models import get_model
from dataset import CLASSES

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR    = "./checkpoints"
MODEL_NAMES = ["resnet", "efficientnet", "convnext"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    print(f"\nPredictions for: {image_path}")
    print("—" * 40)
    for name in MODEL_NAMES:
        model = get_model(name).to(DEVICE)
        model.load_state_dict(torch.load(f"{SAVE_DIR}/{name}_best.pth", map_location=DEVICE))
        model.eval()
        with torch.no_grad():
            probs = torch.softmax(model(tensor), dim=1)[0]
            top3  = probs.topk(3)
        print(f"\n  {name.upper()}")
        for prob, idx in zip(top3.values, top3.indices):
            print(f"    {CLASSES[idx]:<12} {prob.item()*100:6.2f}%")

predict("images.jpeg")

Writing predict.py


In [12]:
!python predict.py images.jpeg


Predictions for: images.jpeg
————————————————————————————————————————

  RESNET
    cat           99.42%
    bird           0.14%
    dog            0.13%

  EFFICIENTNET
    cat           99.96%
    dog            0.02%
    frog           0.01%

  CONVNEXT
    cat           99.92%
    bird           0.07%
    dog            0.01%
